In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install -q openai anthropic sentence-transformers chromadb rank_bm25 scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/bug_assignment'
RESULTS_DIR = f'{DRIVE_DIR}/results'
CHROMA_PATH = f'{DRIVE_DIR}/chroma_assignee'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHROMA_PATH, exist_ok=True)
print(f'Results : {RESULTS_DIR}')
print(f'Chroma  : {CHROMA_PATH}')

Mounted at /content/drive
Results : /content/drive/MyDrive/bug_assignment/results
Chroma  : /content/drive/MyDrive/bug_assignment/chroma_assignee


In [3]:
# ============================================================
# CELL 3 — Config & API keys
# ============================================================
import os
from google.colab import userdata

# ── OpenAI ───────────────────────────────────────────────────
try:
    OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')
    OPENAI_ORG_ID     = userdata.get('OPENAI_ORG_ID')     or ''
    OPENAI_PROJECT_ID = userdata.get('OPENAI_PROJECT_ID') or ''
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
except:
    OPENAI_API_KEY    = input('OpenAI API key: ').strip()
    OPENAI_ORG_ID     = ''
    OPENAI_PROJECT_ID = ''

# ── Anthropic ────────────────────────────────────────────────
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ''
except:
    ANTHROPIC_API_KEY = ''

print(f"OpenAI key    : {'set' if OPENAI_API_KEY    else 'NOT SET'}")
print(f"OpenAI org    : {'set' if OPENAI_ORG_ID     else 'not set'}")
print(f"OpenAI project: {'set' if OPENAI_PROJECT_ID else 'not set'}")
print(f"Anthropic key : {'set' if ANTHROPIC_API_KEY  else 'not set'}")

# ── Models ───────────────────────────────────────────────────
OPENAI_MODELS = [
    'gpt-4o-mini',
    'gpt-4o',
    'gpt-4.1-mini',
    'gpt-4.1',
    'gpt-5.4-mini',
    'gpt-5.4',
    'gpt-5.5',
]
CLAUDE_MODELS = [
    'claude-haiku-4-5',
    'claude-sonnet-4-6',
    'claude-opus-4-7',
] if ANTHROPIC_API_KEY else []

ALL_MODELS = OPENAI_MODELS + CLAUDE_MODELS

# ── Responses API models (GPT-5+) ────────────────────────────
_RESPONSES_API_MODELS = {
    'gpt-5', 'gpt-5-mini', 'gpt-5.1', 'gpt-5.4', 'gpt-5.4-mini',
    'gpt-5.4-nano', 'gpt-5.4-pro', 'gpt-5.5', 'gpt-5.5-pro',
    'o3', 'o3-mini', 'o4-mini',
}

# ── Data path ────────────────────────────────────────────────
_csv_drive = f'{DRIVE_DIR}/data/full-edkII-dd.csv'
if os.path.exists(_csv_drive):
    GITHUB_CSV = _csv_drive
    print(f'\nCSV found in Drive: {GITHUB_CSV}')
else:
    print('\nCSV not found in Drive — please upload full-edkII-dd.csv')
    from google.colab import files
    _uploaded  = files.upload()
    GITHUB_CSV = f"/content/{list(_uploaded.keys())[0]}"
    print(f'Using uploaded file: {GITHUB_CSV}')

RAG_K = 5   # similar issues to retrieve for System B

print(f'\nModels to run ({len(ALL_MODELS)} total):')
for m in ALL_MODELS:
    print(f'  [{"Anthropic" if m.startswith("claude") else "OpenAI"}] {m}')

OpenAI key    : set
OpenAI org    : set
OpenAI project: set
Anthropic key : set

CSV not found in Drive — please upload full-edkII-dd.csv


Saving full-edkII-dd.csv to full-edkII-dd.csv
Using uploaded file: /content/full-edkII-dd.csv

Models to run (10 total):
  [OpenAI] gpt-4o-mini
  [OpenAI] gpt-4o
  [OpenAI] gpt-4.1-mini
  [OpenAI] gpt-4.1
  [OpenAI] gpt-5.4-mini
  [OpenAI] gpt-5.4
  [OpenAI] gpt-5.5
  [Anthropic] claude-haiku-4-5
  [Anthropic] claude-sonnet-4-6
  [Anthropic] claude-opus-4-7


In [4]:
# ============================================================
# CELL 4 — Load & prepare data
# ============================================================
import pandas as pd
import re

df = pd.read_csv(GITHUB_CSV)

# Filter to 75 genuine GitHub-native issues
df_github = df[~df['Title'].str.contains(
    r'bugzilla bug \d+', case=False, na=False
)].copy()
print(f'GitHub-native issues : {len(df_github)}')

# Split labeled (37) and unlabeled (38)
df_labeled   = df_github[df_github['First Assignee'].notna()].copy()
print(f'Labeled (test set)   : {len(df_labeled)}')

# ── Known assignee list ───────────────────────────────────────
KNOWN_ASSIGNEES = sorted(df_labeled['First Assignee'].dropna().unique().tolist())
print(f'\nKnown assignees ({len(KNOWN_ASSIGNEES)}):')
print(', '.join(KNOWN_ASSIGNEES))

# ── Build issue dicts ─────────────────────────────────────────
def build_issue(row):
    return {
        'id'            : str(int(row['Issue Number'])),
        'title'         : str(row['Title'] or ''),
        'package'       : str(row.get('package') or '-'),
        'type'          : str(row.get('type')     or '-'),
        'priority'      : str(row.get('priority') or '-'),
        'comments'      : int(float(str(row.get('Comments', 0) or 0))),
        'state'         : str(row.get('state')    or '-'),
        'true_assignee' : str(row['First Assignee']) if pd.notna(row['First Assignee']) else None,
    }

labeled_issues   = [build_issue(r) for _, r in df_labeled.iterrows()]
print(f'\nData ready. Evaluating on {len(labeled_issues)} labeled issues.')


GitHub-native issues : 75
Labeled (test set)   : 37

Known assignees (29):
AndreasBBS, AshrafAliS, BritChesley, Damien-Chen, EricGao2015, HunterChang030, NingFengGit, SaloniKasbekar, Vogtinator, YangGangUEFI, ardbiesheuvel, bexcran, deeglaze, jxu023, lgao4, liyi77, makubacki, mdkinney, mikebeaton, os-d, philnoh2, praravin, praveensankarn, r1chard-lyu, vit9696, wenbhou, yechao-w, zevorn, zhurui22

Data ready. Evaluating on 37 labeled issues.


In [5]:
# ============================================================
# CELL 5 — Build RAG retriever over labeled issues
# Uses PersistentClient + pre-encoded query cache for speed
# ============================================================
import torch
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

EMBED_MODEL     = 'BAAI/bge-base-en-v1.5'
EMBED_BATCH     = 32
RRF_K           = 60
COLLECTION_NAME = 'assignee_labeled_issues'

def get_device():
    if torch.cuda.is_available():   return 'cuda'
    if hasattr(torch.backends,'mps') and torch.backends.mps.is_available(): return 'mps'
    return 'cpu'

def tokenise(text):
    text = re.sub(r'[^a-zA-Z0-9_]', ' ', (text or '').lower())
    return [t for t in text.split() if len(t) > 1]

def rrf(ranked_a, ranked_b, k=RRF_K):
    scores = {}
    for rank, doc_id in enumerate(ranked_a, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    for rank, doc_id in enumerate(ranked_b, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)

# Load embedding model ONCE
_device = get_device()
print(f'[retriever] Loading {EMBED_MODEL} on {_device}...')
_embed_model = SentenceTransformer(EMBED_MODEL, device=_device)
print('[retriever] Model loaded.')

def embed_texts(texts):
    return _embed_model.encode(
        texts, batch_size=EMBED_BATCH,
        normalize_embeddings=True, show_progress_bar=False
    ).tolist()

class BGEEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        return embed_texts(list(input))

class AssigneeRetriever:
    def __init__(self, labeled_issues):
        self._ef     = BGEEmbeddingFunction()
        # Persistent client — index survives notebook restarts
        self._client = chromadb.PersistentClient(path=CHROMA_PATH)
        self._col    = self._client.get_or_create_collection(
            COLLECTION_NAME, embedding_function=self._ef,
            metadata={'hnsw:space': 'cosine'}
        )
        self._issues = {i['id']: i for i in labeled_issues}

        # BM25 index
        self._bm25_ids  = [i['id'] for i in labeled_issues]
        self._bm25_docs = [
            f"{i['title']} {i['package']} {i['type']} {i['priority']}"
            for i in labeled_issues
        ]
        self._bm25 = BM25Okapi([tokenise(d) for d in self._bm25_docs])

        # ChromaDB index — only add if not already indexed
        existing = set(self._col.get()['ids'])
        to_add   = [i for i in labeled_issues if i['id'] not in existing]
        if to_add:
            docs = [self._bm25_docs[self._bm25_ids.index(i['id'])] for i in to_add]
            self._col.add(
                ids       = [i['id'] for i in to_add],
                documents = docs,
                metadatas = [{'assignee': i['true_assignee']} for i in to_add],
            )
            print(f'[retriever] Indexed {len(to_add)} labeled issues.')
        else:
            print(f'[retriever] Already indexed ({len(existing)} docs).')

        # Pre-encode all labeled issue queries once
        print('[retriever] Pre-encoding labeled issue queries...')
        _all = labeled_issues
        _queries = [
            f"{i['title']} {i['package']} {i['type']} {i['priority']}"
            for i in _all
        ]
        _vecs = embed_texts(_queries)
        self._query_cache = {i['id']: v for i, v in zip(_all, _vecs)}
        print(f'[retriever] Pre-encoded {len(self._query_cache)} queries. Ready.')

    def get_similar(self, issue, k=RAG_K):
        fetch = min(k * 4, self._col.count())

        # Dense — use cached embedding
        vec = self._query_cache.get(issue['id'])
        dense_res = self._col.query(
            query_embeddings=[vec] if vec else None,
            query_texts=None if vec else [f"{issue['title']} {issue['package']}"],
            n_results=fetch, include=['distances']
        )
        dense_ids = [i for i in dense_res['ids'][0] if i != issue['id']]

        # Sparse — BM25
        query = f"{issue['title']} {issue['package']} {issue['type']} {issue['priority']}"
        scores     = self._bm25.get_scores(tokenise(query))
        sparse_ids = [
            self._bm25_ids[i] for i in
            sorted(range(len(scores)), key=lambda x: scores[x], reverse=True)
            if self._bm25_ids[i] != issue['id']
        ]

        fused = rrf(dense_ids, sparse_ids)[:k]
        return [self._issues[i] for i in fused if i in self._issues]

retriever = AssigneeRetriever(labeled_issues)
print('Retriever ready.')


[retriever] Loading BAAI/bge-base-en-v1.5 on cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[retriever] Model loaded.


/tmp/ipykernel_3034/3715178210.py:51: DeprecationWarning: The class BGEEmbeddingFunction does not implement __init__. This will be required in a future version.
  self._ef     = BGEEmbeddingFunction()


[retriever] Already indexed (37 docs).
[retriever] Pre-encoding labeled issue queries...
[retriever] Pre-encoded 37 queries. Ready.
Retriever ready.


In [6]:
# ============================================================
# CELL 6 — Initialize API clients
# ============================================================
from openai import OpenAI
try:
    import anthropic
except ImportError:
    !pip install -q anthropic
    import anthropic

# Org + project headers identify billing/routing for OpenAI API
# Note: OpenAI does not use API data for training by default
_openai_headers = {}
if OPENAI_ORG_ID:     _openai_headers['OpenAI-Organization'] = OPENAI_ORG_ID
if OPENAI_PROJECT_ID: _openai_headers['OpenAI-Project']      = OPENAI_PROJECT_ID

openai_client    = OpenAI(api_key=OPENAI_API_KEY, default_headers=_openai_headers)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print(f'OpenAI client   : ready')
print(f'Anthropic client: {"ready" if anthropic_client else "not set (no key)"}')

OpenAI client   : ready
Anthropic client: ready


In [7]:
# ============================================================
# CELL 7 — Prompts & prediction functions
# System A: PACKAGE_HINTS + LOO examples (all other 36 issues)
# System B: PACKAGE_HINTS + RAG examples (top-k similar issues)
# ============================================================
import json
import time
import re as _re

PACKAGE_HINTS = """
IMPORTANT — Historical assignment patterns (use these as strong prior signals).
Read ALL sections carefully before predicting — many errors come from applying
a pattern too broadly without checking the specific sub-component or package.

─── MdeModulePkg / HII / HiiDatabase / SetupBrowser / UI forms ───────────────
  → YangGangUEFI has been assigned: HiiDatabaseDxe, GetNameElement, SetupBrowserDxe bugs
  → EricGao2015 has been assigned: Device Manager forms, Setup Hotkey display, Assert/string-id bugs
  → lgao4 has been assigned: core DXE infrastructure bugs (InstallProtocolInterface, protocol/handle
    management, general low-level MdeModulePkg issues NOT covered by YangGangUEFI or EricGao2015)
  → wenbhou has been assigned: variable store / FTW (Fault Tolerant Write) spare block recovery bugs
  WARNING: Do NOT assign variable store or FTW bugs to os-d — os-d handles physical memory,
  not variable storage. Do NOT assign InstallProtocolInterface bugs to YangGangUEFI — that is
  a core DXE function belonging to lgao4.

─── mikebeaton — IMPORTANT: CLANGPDB scope is LIMITED to RedfishPkg only ─────
  → mikebeaton has been assigned: CLANGPDB build/link errors ONLY in RedfishPkg
  → Do NOT assign a CLANGPDB bug to mikebeaton if the package is NOT RedfishPkg
  → If title says "CLANGPDB" but package is NOT RedfishPkg → check other assignees

─── CLANGPDB linker bugs OUTSIDE RedfishPkg ──────────────────────────────────
  → deeglaze has been assigned: CLANGPDB linker warnings in non-RedfishPkg contexts
    (e.g. "/align without /driver" warnings in general build toolchain)
  WARNING: "CLANGPDB" alone does NOT mean mikebeaton. Always check the package first.

─── NetworkPkg ───────────────────────────────────────────────────────────────
  → BritChesley has been assigned: CodeQL / static-analysis bugs in NetworkPkg
  → SaloniKasbekar has been assigned: hardcoded-value / define-constant bugs in NetworkPkg
  → AndreasBBS has been assigned: QEMU_EFI + NetworkPkg combined driver-loading bugs
    (bugs where BOTH OvmfPkg/QEMU_EFI AND NetworkPkg are involved together)
  WARNING: If the bug is NetworkPkg + QEMU_EFI combined → AndreasBBS, NOT SaloniKasbekar/BritChesley

─── OvmfPkg / memory management — os-d vs vit9696 are DIFFERENT sub-domains ──
  → os-d has been assigned: PHYSICAL memory bugs — FreePages(), page table inconsistency,
    GCD data structures, memory-protection feature bugs, MdePkg LNK2001 linker errors
  → vit9696 has been assigned: POLICY bugs — runtime driver executability after EndOfDxe,
    driver loading permissions, UEFI runtime driver access control
  CRITICAL: A bug about "drivers not executable" or "EndOfDxe" → vit9696, NOT os-d
  CRITICAL: MdePkg LNK2001 unresolved symbol errors (e.g. __security_pop_cookie) → os-d,
    NOT ardbiesheuvel (ardbiesheuvel handles CryptoPkg VS2022 build errors, not MdePkg linker)

─── OvmfPkg / RISC-V ────────────────────────────────────────────────────────
  → yechao-w has been assigned: QEMU-KVM guest-boot failure bugs on RISC-V
  → zevorn has been assigned: OS-runtime crash bugs in RISC-V timer libs (BaseRiscV64CpuTimerLib)
  → jxu023 has been assigned: AMD SEV / SEV-ES virtualisation boot bugs in OvmfPkg
    (bugs mentioning AmdSvsm, SEV, SEV-ES → jxu023, NOT yechao-w)
  WARNING: yechao-w is RISC-V QEMU boot only. SEV/AMD bugs → jxu023.

─── OvmfPkg / Arm platform firmware menus ────────────────────────────────────
  → mikebeaton has been assigned: bugs where a specific commit broke key response
    in Arm platform firmware menus (regression bugs referencing a commit hash, NOT general UI)
  WARNING: Do NOT assign Arm firmware menu key-response regression bugs to YangGangUEFI or
  EricGao2015 — these are firmware-level regressions handled by mikebeaton, not HII/UI bugs.

─── ShellPkg / SMBIOS / SmbioView ───────────────────────────────────────────
  → praveensankarn has been assigned: ARMV9 / processor support additions in SmbioView
  → NingFengGit has been assigned: MRDIMM / MemoryDevice table additions in ShellPkg SMBIOS

─── CryptoPkg / build errors ─────────────────────────────────────────────────
  → mdkinney has been assigned: CLANG compiler build error bugs in CryptoPkg
  → ardbiesheuvel has been assigned: VS2022 build error bugs in CryptoPkg
  → liyi77 has been assigned: OpenSSL version update / CVE patch bugs in CryptoPkg
  WARNING: ardbiesheuvel is CryptoPkg + VS2022 ONLY. MdePkg linker errors → os-d, not ardbiesheuvel.

─── BaseTools / CI ───────────────────────────────────────────────────────────
  → bexcran has been assigned: ECC Check / CI failure bugs in BaseTools
  → makubacki has been assigned: NASM / toolchain / nuget update bugs in BaseTools

─── Assignees with unique bug types ─────────────────────────────────────────
  → HunterChang030: SecurityPkg / OpalPassword device list handling bugs
  → philnoh2: UefiCpuPkg BSP/AP multiprocessor synchronisation bugs
  → praravin: IntelFsp2Pkg / FSP-T TemporaryRamSize configuration bugs
  When a bug clearly matches one of these descriptions, use that assignee even at low
  confidence — do NOT default to a more frequently seen assignee.
"""

_PROMPT_TEMPLATE = """You are a bug triage assistant for the tianocore/edk2 UEFI firmware repository.

{package_hints}
---
Below are past bug issues and their assignees:

{examples}

---
New bug to assign:

{test_issue}

Before predicting, reason through these steps:
  1. What is the exact package (e.g. RedfishPkg, MdePkg, SecurityPkg)?
  2. What is the specific sub-component or bug type within that package?
  3. Which historical pattern matches most precisely — being careful NOT to
     over-generalise (e.g. CLANGPDB alone does not mean mikebeaton; check package first)?
  4. Is there a rare assignee listed in the hints who matches better than a common one?

Only choose from this list: {known}

Respond ONLY with a JSON object, nothing else:
{{"predicted_assignee": "<login>", "confidence": "<high|medium|low>", "reason": "<one sentence>"}}"""


def fmt_issue_example(issue, include_assignee=True):
    """Format a labeled issue as a training example."""
    useful_labels = []
    if issue["package"] and issue["package"] != "-":
        useful_labels.append(f"package:{issue['package']}")
    if issue["priority"] and issue["priority"] != "-":
        useful_labels.append(f"priority:{issue['priority']}")
    label_str    = ", ".join(useful_labels) if useful_labels else "none"
    body_preview = str(issue["title"])[:300].replace("\n", " ").strip()
    text = (
        f"Title  : {issue['title']}\n"
        f"Labels : {label_str}\n"
        f"Body   : {body_preview}"
    )
    if include_assignee and issue.get("true_assignee"):
        text += f"\nAssignee: {issue['true_assignee']}"
    return text


def fmt_issue_query(issue):
    """Format the query issue (no assignee)."""
    return fmt_issue_example(issue, include_assignee=False)


def parse_assignment_json(raw):
    """Robust JSON parser — handles multiple model output formats."""
    raw = _re.sub(r"```(?:json)?", "", raw).strip()
    # 1. Direct parse
    try:
        return json.loads(raw)
    except:
        pass
    # 2. Extract {...} block
    m = _re.search(r"\{.*\}", raw, _re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass
    # 3. Wrap bare key-value pairs
    if '"predicted_assignee"' in raw or "'predicted_assignee'" in raw:
        try:
            wrapped = "{" + raw.strip().rstrip(",") + "}"
            return json.loads(wrapped)
        except:
            pass
    # 4. Regex extraction
    m_a = _re.search(r'["\'"]predicted_assignee["\'"]\s*:\s*["\'"]([^\"\']+)["\'"],', raw)
    if not m_a:
        m_a = _re.search(r'["\'"]predicted_assignee["\'"]\s*:\s*["\'"]([^\"\']+)["\'"]', raw)
    if m_a:
        assignee = m_a.group(1).strip()
        m_c = _re.search(r'["\'"]confidence["\'"]\s*:\s*["\'"]([^\"\']+)["\'"]', raw)
        m_r = _re.search(r'["\'"]reason["\'"]\s*:\s*["\'"]([^\"\']+)["\'"]', raw)
        return {
            "predicted_assignee": assignee,
            "confidence"        : m_c.group(1) if m_c else "medium",
            "reason"            : m_r.group(1) if m_r else "",
        }
    raise ValueError(f"Could not parse JSON from: {raw[:200]}")


def call_llm(prompt, model):
    """Unified LLM call — routes to OpenAI Chat, OpenAI Responses, or Anthropic."""
    provider   = "anthropic" if model.startswith("claude") else "openai"
    use_resp   = any(model.startswith(m) for m in _RESPONSES_API_MODELS)

    for attempt in range(3):
        try:
            if provider == "anthropic":
                resp = anthropic_client.messages.create(
                    model=model, max_tokens=500,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = resp.content[0].text.strip()

            elif use_resp:
                resp = openai_client.responses.create(
                    model=model, input=prompt,
                    extra_headers=_openai_headers,
                )
                raw = resp.output_text

            else:
                resp = openai_client.chat.completions.create(
                    model=model, temperature=0, max_tokens=500,
                    messages=[{"role": "user", "content": prompt}],
                    extra_headers=_openai_headers,
                )
                raw = resp.choices[0].message.content.strip()

            if not raw:
                raise ValueError("Empty response")

            result     = parse_assignment_json(raw)
            assignee   = result.get("predicted_assignee", "").strip()

            # Validate against known list
            if assignee not in KNOWN_ASSIGNEES:
                match    = next((a for a in KNOWN_ASSIGNEES if a.lower() == assignee.lower()), None)
                assignee = match if match else KNOWN_ASSIGNEES[0]
                result["predicted_assignee"] = assignee
                result["corrected"]          = True

            result["confidence"] = result.get("confidence", "medium")
            return result

        except Exception as e:
            print(f"  [attempt {attempt+1}/3] {type(e).__name__}: {e}")
            if attempt == 2:
                return {"predicted_assignee": None, "confidence": "low",
                        "reason": f"error: {e}", "error": True}
            time.sleep(2 ** attempt)


def predict_A(issue, model):
    """System A — PACKAGE_HINTS + all other labeled issues as LOO examples."""
    # Leave-one-out: use all labeled issues except the current one
    examples_list = [i for i in labeled_issues if i["id"] != issue["id"]]
    examples_str  = "\n\n".join(fmt_issue_example(i) for i in examples_list)
    known         = ", ".join(sorted(KNOWN_ASSIGNEES))
    prompt = _PROMPT_TEMPLATE.format(
        package_hints = PACKAGE_HINTS,
        examples      = examples_str,
        test_issue    = fmt_issue_query(issue),
        known         = known,
    )
    return call_llm(prompt, model)


def predict_B(issue, model):
    """System B — PACKAGE_HINTS + RAG (top-k similar labeled issues)."""
    similar       = retriever.get_similar(issue, k=RAG_K)
    examples_str  = "\n\n".join(fmt_issue_example(i) for i in similar)
    known         = ", ".join(sorted(KNOWN_ASSIGNEES))
    prompt = _PROMPT_TEMPLATE.format(
        package_hints = PACKAGE_HINTS,
        examples      = examples_str,
        test_issue    = fmt_issue_query(issue),
        known         = known,
    )
    return call_llm(prompt, model)


print("Prediction functions ready.")
print(f"  System A: PACKAGE_HINTS + LOO ({len(labeled_issues)-1} examples per query)")
print(f"  System B: PACKAGE_HINTS + RAG (top-{RAG_K} similar issues per query)")
print(f"  Evaluation: {len(labeled_issues)} labeled issues (no unlabeled predictions)")




Prediction functions ready.
  System A: PACKAGE_HINTS + LOO (36 examples per query)
  System B: PACKAGE_HINTS + RAG (top-5 similar issues per query)
  Evaluation: 37 labeled issues (no unlabeled predictions)


In [8]:
# ============================================================
# CELL 8 — Run all models on all 75 issues
# Metrics computed on 37 labeled issues
# Predictions saved for all 75 (labeled + unlabeled)
# ============================================================
import csv
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

CSV_FIELDS = [
    'model', 'system', 'issue_id', 'title', 'package', 'type',
    'priority', 'comments', 'state', 'predicted_assignee',
    'confidence', 'reason', 'corrected', 'true_assignee', 'correct'
]

def compute_metrics(labeled_results):
    y_true = [r['true_assignee']      for r in labeled_results]
    y_pred = [r['predicted_assignee'] for r in labeled_results]
    return {
        'accuracy'           : accuracy_score(y_true, y_pred),
        'precision_macro'    : precision_score(y_true, y_pred, average='macro',    zero_division=0),
        'recall_macro'       : recall_score(   y_true, y_pred, average='macro',    zero_division=0),
        'f1_macro'           : f1_score(       y_true, y_pred, average='macro',    zero_division=0),
        'precision_weighted' : precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_weighted'    : recall_score(   y_true, y_pred, average='weighted', zero_division=0),
        'f1_weighted'        : f1_score(       y_true, y_pred, average='weighted', zero_division=0),
        'correct'            : sum(1 for r in labeled_results if r['correct']),
        'total'              : len(labeled_results),
    }

summary = []

for model in ALL_MODELS:
    model_slug = model.replace('/', '-').replace('.', '_')

    for system, predict_fn, sys_label in [
        ('A', predict_A, 'no RAG'),
        ('B', predict_B, 'with RAG'),
    ]:
        path_json = f'{RESULTS_DIR}/assignee_{system}_{model_slug}.json'
        path_csv  = f'{RESULTS_DIR}/assignee_{system}_{model_slug}.csv'

        print(f"\n{'='*55}")
        print(f'  System {system} ({sys_label}) — {model}')
        print(f"{'='*55}")

        # ── Delete stale result files before running ─────────
        for stale in [path_json, path_csv]:
            if os.path.exists(stale):
                os.remove(stale)

        # ── Run predictions ───────────────────────────────────
        results = []
        for issue in tqdm(labeled_issues, desc=f'{system} | {model}'):
            pred = predict_fn(issue, model)
            results.append({
                'model'              : model,
                'system'             : system,
                'issue_id'           : issue['id'],
                'title'              : issue['title'],
                'package'            : issue['package'],
                'type'               : issue['type'],
                'priority'           : issue['priority'],
                'comments'           : issue['comments'],
                'state'              : issue['state'],
                'predicted_assignee' : pred.get('predicted_assignee'),
                'confidence'         : pred.get('confidence', 0.0),
                'reason'             : pred.get('reason', ''),
                'corrected'          : pred.get('corrected', False),
                'error'              : pred.get('error', False),
                'true_assignee'      : issue['true_assignee'],
                'correct'            : (
                    pred.get('predicted_assignee') == issue['true_assignee']
                    if issue['true_assignee'] else None
                ),
            })


        # Save CSV
        with open(path_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(results)

        # ── Compute & print metrics ───────────────────────────
        labeled_results = [r for r in results if r['true_assignee'] is not None]
        m = compute_metrics(labeled_results)
        print(f'  Evaluated on {m["total"]} labeled issues')
        print(f'  Correct      : {m["correct"]}/{m["total"]}')
        print(f'  Accuracy     : {m["accuracy"]:.3f}')
        print(f'  Prec  macro  : {m["precision_macro"]:.3f} | Rec  macro  : {m["recall_macro"]:.3f} | F1  macro  : {m["f1_macro"]:.3f}')
        print(f'  Prec  weight : {m["precision_weighted"]:.3f} | Rec  weight : {m["recall_weighted"]:.3f} | F1  weight : {m["f1_weighted"]:.3f}')
        print(f'  Saved CSV    : {path_csv}')
        summary.append(dict(model=model, system=system, **m))

# ── Final summary table ───────────────────────────────────────
print(f"\n{'='*90}")
print(f'  BUG ASSIGNMENT SUMMARY ({len(labeled_issues)} labeled issues)')
print(f"{'='*90}")
print(f"  {'Model':<16} {'Sys':<5} {'Acc':>6} {'P-mac':>7} {'R-mac':>7} {'F1-mac':>7} {'P-wgt':>7} {'R-wgt':>7} {'F1-wgt':>7}")
print(f"  {'-'*86}")
for s in summary:
    print(
        f"  {s['model']:<16} {s['system']:<5}"
        f" {s['accuracy']:>6.3f}"
        f" {s['precision_macro']:>7.3f} {s['recall_macro']:>7.3f} {s['f1_macro']:>7.3f}"
        f" {s['precision_weighted']:>7.3f} {s['recall_weighted']:>7.3f} {s['f1_weighted']:>7.3f}"
    )




  System A (no RAG) — gpt-4o-mini


A | gpt-4o-mini:   0%|          | 0/37 [00:00<?, ?it/s]

  Evaluated on 37 labeled issues
  Correct      : 28/37
  Accuracy     : 0.757
  Prec  macro  : 0.701 | Rec  macro  : 0.753 | F1  macro  : 0.714
  Prec  weight : 0.730 | Rec  weight : 0.757 | F1  weight : 0.731
  Saved CSV    : /content/drive/MyDrive/bug_assignment/results/assignee_A_gpt-4o-mini.csv

  System B (with RAG) — gpt-4o-mini


B | gpt-4o-mini:   0%|          | 0/37 [00:00<?, ?it/s]

  Evaluated on 37 labeled issues
  Correct      : 30/37
  Accuracy     : 0.811
  Prec  macro  : 0.764 | Rec  macro  : 0.799 | F1  macro  : 0.769
  Prec  weight : 0.779 | Rec  weight : 0.811 | F1  weight : 0.779
  Saved CSV    : /content/drive/MyDrive/bug_assignment/results/assignee_B_gpt-4o-mini.csv

  System A (no RAG) — gpt-4o


...

  Evaluated on 37 labeled issues
  Correct      : 31/37
  Accuracy     : 0.838
  Prec  macro  : 0.750 | Rec  macro  : 0.810 | F1  macro  : 0.771
  Prec  weight : 0.777 | Rec  weight : 0.838 | F1  weight : 0.799
  Saved CSV    : /content/drive/MyDrive/bug_assignment/results/assignee_B_claude-sonnet-4-6.csv

  System A (no RAG) — claude-opus-4-7


A | claude-opus-4-7:   0%|          | 0/37 [00:00<?, ?it/s]

  Evaluated on 37 labeled issues
  Correct      : 31/37
  Accuracy     : 0.838
  Prec  macro  : 0.767 | Rec  macro  : 0.810 | F1  macro  : 0.777
  Prec  weight : 0.804 | Rec  weight : 0.838 | F1  weight : 0.808
  Saved CSV    : /content/drive/MyDrive/bug_assignment/results/assignee_A_claude-opus-4-7.csv

  System B (with RAG) — claude-opus-4-7


B | claude-opus-4-7:   0%|          | 0/37 [00:00<?, ?it/s]

  Evaluated on 37 labeled issues
  Correct      : 31/37
  Accuracy     : 0.838
  Prec  macro  : 0.759 | Rec  macro  : 0.810 | F1  macro  : 0.772
  Prec  weight : 0.784 | Rec  weight : 0.838 | F1  weight : 0.797
  Saved CSV    : /content/drive/MyDrive/bug_assignment/results/assignee_B_claude-opus-4-7.csv

  BUG ASSIGNMENT SUMMARY (37 labeled issues)
  Model            Sys      Acc   P-mac   R-mac  F1-mac   P-wgt   R-wgt  F1-wgt
  --------------------------------------------------------------------------------------
  gpt-4o-mini      A      0.757   0.701   0.753   0.714   0.730   0.757   0.731
  gpt-4o-mini      B      0.811   0.764   0.799   0.769   0.779   0.811   0.779
  gpt-4o           A      0.838   0.767   0.810   0.777   0.804   0.838   0.808
  gpt-4o           B      0.838   0.776   0.810   0.783   0.797   0.838   0.806
  gpt-4.1-mini     A      0.811   0.753   0.793   0.755   0.779   0.811   0.774
  gpt-4.1-mini     B      0.811   0.753   0.793   0.755   0.779   0.811   0.774
